# Taxi Demand Forecasting
**Goal:** Predict the number of taxi orders for the next hour at airports, with RMSE ≤ 48 on the test set.

**Dataset:** Historical taxi orders collected at 10-minute intervals (March–August 2018).

## 1. Data Loading & Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
df = pd.read_csv('/datasets/taxi.csv')

# Parse datetime and set as index
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.set_index('datetime')

# Resample to hourly frequency
df_hourly = df.resample('1H').sum()

print(f'Shape after resampling: {df_hourly.shape}')
print(f'Period: {df_hourly.index.min()} to {df_hourly.index.max()}')
df_hourly.head()

**No missing values or duplicate timestamps were found.** Values repeated in `num_orders` across different timestamps are expected in a count series and are not duplicates.

## 2. Exploratory Data Analysis

In [ ]:
df_hourly['num_orders'].plot(figsize=(14, 4), title='Hourly Taxi Orders (March–August 2018)')
plt.ylabel('Number of Orders')
plt.tight_layout()
plt.show()

In [ ]:
decomposed = seasonal_decompose(df_hourly['num_orders'], period=24)

fig, axes = plt.subplots(3, 1, figsize=(14, 8))
decomposed.trend.plot(ax=axes[0], title='Trend')
decomposed.seasonal.plot(ax=axes[1], title='Seasonality (24h)')
decomposed.resid.plot(ax=axes[2], title='Residuals')
plt.tight_layout()
plt.show()

In [ ]:
hourly_avg = df_hourly.copy()
hourly_avg['hour'] = hourly_avg.index.hour
hourly_avg.groupby('hour')['num_orders'].mean().plot(
    figsize=(10, 4),
    title='Average Orders by Hour of Day',
    marker='o'
)
plt.xlabel('Hour')
plt.ylabel('Average Orders')
plt.tight_layout()
plt.show()

**Key findings:**
- Demand shows a consistent **upward trend** from March to August.
- Clear **daily seasonality**: demand drops between 4–6 AM and peaks at **4–6 PM** and again at **8–11 PM**.
- Residuals show unexplained variance, likely driven by specific events or outliers.

## 3. Feature Engineering

In [ ]:
df_feat = df_hourly.copy()

# Calendar features
df_feat['hour'] = df_feat.index.hour
df_feat['day_of_week'] = df_feat.index.dayofweek

# Lag features (1h to 48h)
for lag in range(1, 49):
    df_feat[f'lag_{lag}'] = df_feat['num_orders'].shift(lag)

# Rolling mean features (shift=1 to prevent data leakage)
df_feat['rolling_mean_24'] = df_feat['num_orders'].shift(1).rolling(24).mean()
df_feat['rolling_mean_48'] = df_feat['num_orders'].shift(1).rolling(48).mean()

# Drop NaN rows created by lags
df_feat = df_feat.dropna()

print(f'Features created: {df_feat.shape[1] - 1} | Rows after dropna: {df_feat.shape[0]}')

## 4. Model Training

In [ ]:
X = df_feat.drop('num_orders', axis=1)
y = df_feat['num_orders']

# Chronological split — no shuffle to respect time series order
split_index = int(len(df_feat) * 0.9)
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results[name] = rmse
    status = '✅' if rmse <= 48 else '❌'
    print(f'{status} {name}: RMSE = {rmse:.2f}')

## 5. Conclusion

Three models were trained to predict hourly taxi demand: **Linear Regression**, **Random Forest**, and **Gradient Boosting**.

| Model | RMSE | Within Limit (≤ 48)? |
|---|---|---|
| Linear Regression | 42.86 | ✅ |
| Random Forest | 47.09 | ✅ |
| Gradient Boosting | 63.17 | ❌ |

**Best model: Linear Regression** with RMSE ≈ 42.86, well below the required threshold of 48.

The strong performance of Linear Regression suggests that **lag features effectively capture the linear temporal dependency** in this dataset. The model can be used in production to anticipate demand spikes and support driver allocation decisions during peak hours.